# Jena Climate Forecasting

Predict the next **7 observations** of temperature from 144 past multivariate observations, using Conv1D and LSTM layers.

Original saved results: model MAE **0.452°C**, persistence baseline MAE **0.506°C**. These are historical results, not a fresh training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from data import load_climate, prepare_data, build_datasets
from model import build_model
from train import evaluate_naive, train_model, evaluate_model


## 1. Load the dataset


In [ ]:
df = load_climate()
print("Dataset shape:", df.shape)
df.head()


## 2. Prepare features and chronological splits


In [ ]:
data = prepare_data(df)
print("Train:", data.num_train, "Validation:", data.num_val,
      "Test:", len(data.features) - data.num_train - data.num_val)


## 3. Build supervised windows


In [ ]:
lookback, horizon, batch_size = 144, 7, 256
train_ds, val_ds, test_ds = build_datasets(data, lookback, horizon, batch_size)


## 4. Evaluate the persistence baseline


In [ ]:
naive_mae = evaluate_naive(test_ds, data, horizon)
print(f"Naive baseline MAE: {naive_mae:.3f} °C")


## 5. Build the Conv1D + LSTM model


In [ ]:
model = build_model(data.features.shape[1], lookback, horizon)
model.summary()


## 6. Train with early stopping


In [ ]:
history = train_model(model, train_ds, val_ds, epochs=20)


## 7. Evaluate on the test split


In [ ]:
results = evaluate_model(model, test_ds)
print(f"Test MAE: {results['mae']:.3f} °C")


## 8. Learning curves


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["mae"], label="Train")
plt.plot(history.history["val_mae"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("MAE (°C)")
plt.title("Temperature forecasting learning curves")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 9. Example forecast

One example illustrates the seven forecast steps; it is not an aggregate measure of performance.


In [ ]:
x_batch, y_batch = next(iter(test_ds))
predicted = model(x_batch[:1], training=False).numpy()[0]
last_temp = float(x_batch[0, -1, data.temp_col] * data.std[data.temp_col] + data.mean[data.temp_col])
steps = np.arange(1, horizon + 1)
plt.plot(steps, y_batch[0].numpy(), "o-", label="Observed")
plt.plot(steps, predicted, "o-", label="Conv1D + LSTM")
plt.axhline(last_temp, linestyle="--", label="Persistence baseline")
plt.xlabel("Future observation step")
plt.ylabel("Temperature (°C)")
plt.xticks(steps)
plt.legend()
plt.show()


## Provenance and limitations

Developed for DATAX504 Assignment 5, time-series track. The original notebook disclosed Codex GPT-5.5 assistance with suggestions and explanations, followed by author review and execution. Codex assisted in extracting this route, correcting the download path for the inspected Keras version, documenting it and adding visualisations. Full training was not repeated during packaging.

Inputs are normalised using training statistics; targets stay in °C. Windows stay within their respective 50%/25%/25% chronological splits. Only training windows are shuffled. No global random seed was set in the original experiment. This is a seven-observation forecast, not a seven-day forecast.
